# Generate 15 bp barcodes with Hamming distance >= 3

- Generate 24,000 barcodes
- Barcode length: 15 bp
- Minimum pairwise Hamming distance: 3
- Max GC fraction: 0.70
- Exclude barcodes containing listed restriction enzyme sites (both strands)
- Random seed is fixed for reproducibility

---

## 腳本說明

**目的**：產生一組互相區隔良好的 DNA barcode（條碼），供啟動子文庫的樣本標記／定序後 demultiplex 使用。任兩條條碼至少相差 `MIN_HAMMING` 個鹼基，讓定序或合成錯誤不容易把一條條碼誤讀成另一條。

**產出**：CSV 檔 `../tables/barcodes_15bp_HD3_no_RE_sites.csv`，共 `N_BARCODES` 列，欄位：
- `barcode_id` — 流水號（`BC00001`、`BC00002` …）
- `barcode` — 15 bp 序列（僅 A/C/G/T）
- `gc_fraction` — 該序列的 GC 比例

**生成方法**：隨機抽一條序列 → 依序過濾（排除限制酶位點、GC 比例超標）→ 用「Hamming 半徑內鄰居查表」判斷是否與**已收條碼**太接近，全部通過才收下；重複到湊滿 `N_BARCODES` 條。整段流程用固定亂數種子，結果可完全重現。

**可調整變數（都在下一格 code）**：
- `N_BARCODES` — 要產生的條碼數（目前 24,000）
- `BARCODE_LEN` — 條碼長度 bp（目前 15）
- `MIN_HAMMING` — 任兩條最小 Hamming distance（目前 3）；`CONFLICT_RADIUS` 會自動 = `MIN_HAMMING - 1`
- `MAX_GC_FRACTION` — GC 比例上限（目前 0.70）
- `RANDOM_SEED` — 亂數種子；換掉會得到不同、但同樣合格的一組條碼
- `MAX_ATTEMPTS` — 嘗試次數上限；湊不滿會直接報錯
- `OUTPUT_PATH` — 輸出檔路徑（改規格時記得一起改，避免蓋到別組結果）
- `RE_SITES` — 要排除的限制酶辨識位點表（在下下格）

> 註：調小 `MIN_HAMMING` 或 `BARCODE_LEN`、調高 `MAX_GC_FRACTION` 會讓合格空間變大、生成更快；反之會變慢，極端時可能湊不滿而觸發 `MAX_ATTEMPTS` 報錯。


## 1. 參數設定與匯入

設定所有可調參數並建立亂數產生器。要改條碼規格（數量、長度、HD、GC 上限、種子、輸出路徑）都集中在這一格。

- `CONFLICT_RADIUS = MIN_HAMMING - 1`：後面判斷「太接近」用的半徑，改 `MIN_HAMMING` 就會自動連動。
- `rng = random.Random(RANDOM_SEED)`：獨立亂數器，固定種子確保結果可重現。
- `BASES = 'ACGT'`：條碼使用的鹼基字元集。


In [ ]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


In [ ]:
from itertools import combinations
from pathlib import Path
import random

import pandas as pd

N_BARCODES = 24_000
BARCODE_LEN = 15
MIN_HAMMING = 3
CONFLICT_RADIUS = MIN_HAMMING - 1
MAX_GC_FRACTION = 0.70
RANDOM_SEED = 42
MAX_ATTEMPTS = 500_000_000

# Derived from the parameters above, never hardcoded: the previous literal
# filename is how a 12 bp run came to be stored as 'barcodes_15bp_HD3'.
# Writes into the release outputs/ folder; tables/ is left untouched.
SPEC_NAME = f'barcodes_{BARCODE_LEN}bp_HD{MIN_HAMMING}_no_RE_sites'
OUTPUT_PATH = RELEASE_OUT / '05_barcodes.csv'

rng = random.Random(RANDOM_SEED)
BASES = 'ACGT'


## 2. 限制酶位點表與輔助函式

定義後續選殖要避開的限制酶辨識序列，以及過濾用的小工具。

- `RE_SITES`：限制酶名稱 → 辨識序列的對照表。
- `revcomp(seq)`：回傳序列的反向互補股。
- `has_re_site(seq)`：檢查序列**本身與其反向互補股**是否含任一位點（雙股都查），有就淘汰。
- `random_barcode(length)`：隨機抽一條長度為 `length` 的序列。
- `gc_fraction(seq)` / `passes_gc_filter(seq)`：計算 GC 比例，並判斷是否 ≤ `MAX_GC_FRACTION`。


In [32]:
# Restriction enzyme recognition sites to avoid.
RE_SITES = {
    'BamHI':   'GGATCC',
    'BcuI':    'ACTAGT',   # = SpeI
    'BglII':   'AGATCT',
    'Eco31I':  'GGTCTC',   # = BsaI
    'EcoRI':   'GAATTC',
    'HindIII': 'AAGCTT',
    'KpnI':    'GGTACC',
    'MluI':    'ACGCGT',
    'NcoI':    'CCATGG',
    'NdeI':    'CATATG',
    'NheI':    'GCTAGC',
    'NotI':    'GCGGCCGC',
    'PstI':    'CTGCAG',
    'SacI':    'GAGCTC',
    'SalI':    'GTCGAC',
    'SmaI':    'CCCGGG',
    'VspI':    'ATTAAT',   # = AseI
    'XbaI':    'TCTAGA',
    'XhoI':    'CTCGAG',
}

_comp = str.maketrans('ACGT', 'TGCA')

def revcomp(seq):
    return seq.translate(_comp)[::-1]

def has_re_site(seq, re_sites=RE_SITES, check_revcomp=True):
    seq = str(seq).upper()
    targets = [seq]
    if check_revcomp:
        targets.append(revcomp(seq))

    for target in targets:
        for site in re_sites.values():
            if site in target:
                return True
    return False

def random_barcode(length=BARCODE_LEN):
    return ''.join(rng.choices(BASES, k=length))

def gc_fraction(seq):
    seq = str(seq).upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

def passes_gc_filter(seq, max_gc_fraction=MAX_GC_FRACTION):
    return gc_fraction(seq) <= max_gc_fraction


## 3. Hamming 鄰居枚舉（核心去重邏輯）

`neighbors_within_radius(seq, radius)`：列出所有與 `seq` 的 Hamming distance ≤ `radius` 的序列（含 `seq` 自己）。

主迴圈用它反查——如果候選條碼的任一個鄰居**已經被收過**，代表兩者距離 ≤ `radius`（即 < `MIN_HAMMING`），就淘汰。這樣就能保證留下來的條碼**任兩條距離 ≥ `MIN_HAMMING`**。

作法：對每個距離 `d`（1 到 `radius`），列舉要更動的 `d` 個位置組合，把這些位置換成其他鹼基後逐一 `yield`。半徑越大、序列越長，枚舉數量增長很快（所以 HD 越大跑越慢）。


In [33]:
def neighbors_within_radius(seq, radius=CONFLICT_RADIUS):
    seq = list(seq)
    length = len(seq)

    if radius < 0:
        return

    yield ''.join(seq)

    for distance in range(1, radius + 1):
        for positions in combinations(range(length), distance):
            originals = [seq[pos] for pos in positions]

            def mutate_position(k):
                if k == distance:
                    yield ''.join(seq)
                    return

                pos = positions[k]
                original = originals[k]
                for base in BASES:
                    if base == original:
                        continue
                    seq[pos] = base
                    yield from mutate_position(k + 1)
                seq[pos] = original

            yield from mutate_position(0)


## 4. 主生成迴圈

反覆隨機抽候選條碼，依序通過三道關卡才收下：

1. **不含限制酶位點**（雙股，`has_re_site`）
2. **GC 比例合格**（`passes_gc_filter`，≤ `MAX_GC_FRACTION`）
3. **與已收條碼距離夠遠**（`neighbors_within_radius` 反查，確保 ≥ `MIN_HAMMING`）

每收滿 1,000 條印一次進度（含累計嘗試次數）。收滿 `N_BARCODES` 或超過 `MAX_ATTEMPTS` 就停；若沒收滿會 `raise RuntimeError`。最後整理成 DataFrame，附上 `barcode_id` 與 `gc_fraction`。


In [34]:
accepted = []
accepted_set = set()
attempts = 0

while len(accepted) < N_BARCODES and attempts < MAX_ATTEMPTS:
    attempts += 1
    candidate = random_barcode()

    if has_re_site(candidate):
        continue

    if not passes_gc_filter(candidate):
        continue

    if any(neighbor in accepted_set for neighbor in neighbors_within_radius(candidate)):
        continue

    accepted.append(candidate)
    accepted_set.add(candidate)

    if len(accepted) % 1000 == 0:
        print(f'{len(accepted)} / {N_BARCODES}, attempts={attempts}')

if len(accepted) < N_BARCODES:
    raise RuntimeError(f'Only generated {len(accepted)} barcodes after {attempts} attempts')

barcodes = pd.DataFrame({
    'barcode_id': [f'BC{i + 1:05d}' for i in range(len(accepted))],
    'barcode': accepted,
})
barcodes['gc_fraction'] = [gc_fraction(seq) for seq in barcodes['barcode']]

print(f'Generated {len(barcodes)} barcodes after {attempts} attempts')
barcodes.head()


1000 / 24000, attempts=1119
2000 / 24000, attempts=2259
3000 / 24000, attempts=3416
4000 / 24000, attempts=4587
5000 / 24000, attempts=5755
6000 / 24000, attempts=6939
7000 / 24000, attempts=8160
8000 / 24000, attempts=9380
9000 / 24000, attempts=10589
10000 / 24000, attempts=11834
11000 / 24000, attempts=13085
12000 / 24000, attempts=14388
13000 / 24000, attempts=15715
14000 / 24000, attempts=17041
15000 / 24000, attempts=18421
16000 / 24000, attempts=19778
17000 / 24000, attempts=21182
18000 / 24000, attempts=22600
19000 / 24000, attempts=24061
20000 / 24000, attempts=25490
21000 / 24000, attempts=26946
22000 / 24000, attempts=28442
23000 / 24000, attempts=29925
24000 / 24000, attempts=31458
Generated 24000 barcodes after 31458 attempts


,barcode_id,barcode,gc_fraction
0,BC00001,GACAGGTACAAGAAG,0.466667
1,BC00002,GAGTATGCATCAATG,0.400000
2,BC00003,TGGTCGTGTGGAACA,0.533333
3,BC00004,AACGCCACTGGAGAC,0.600000
4,BC00005,TGGGTTAACCATTCG,0.466667


## 5. 驗證

對產出結果做完整自我檢查，任一項不過就 `raise`：

- 數量正確、序列**唯一無重複**
- 長度都是 `BARCODE_LEN`、字元只含 A/C/G/T
- 全部**無限制酶位點**、**GC 合格**
- 用鄰居查表做**完整的 Hamming distance 驗證**，確認任兩條距離都 ≥ `MIN_HAMMING`

（這格是獨立於生成迴圈的把關，避免邏輯有漏。）


In [35]:
barcode_set = set(barcodes['barcode'])

assert len(barcodes) == N_BARCODES
assert len(barcode_set) == N_BARCODES
assert all(len(seq) == BARCODE_LEN for seq in barcodes['barcode'])
assert all(set(seq) <= set(BASES) for seq in barcodes['barcode'])
assert all(not has_re_site(seq) for seq in barcodes['barcode'])
assert all(passes_gc_filter(seq) for seq in barcodes['barcode'])

# Full Hamming distance validation using radius lookup.
for seq in barcodes['barcode']:
    barcode_set.remove(seq)
    has_conflict = any(neighbor in barcode_set for neighbor in neighbors_within_radius(seq))
    barcode_set.add(seq)
    if has_conflict:
        raise ValueError(f'Hamming distance conflict found around {seq}')

print(f'Validation passed: unique, no RE sites, full HD >= {MIN_HAMMING}')


Validation passed: unique, no RE sites, full HD >= 4


## 6. 輸出存檔

若輸出資料夾不存在則建立，並把條碼表寫成 CSV 到 `OUTPUT_PATH`。

⚠️ 目標檔 `barcodes_15bp_HD3_no_RE_sites.csv` 若已存在會被**直接覆蓋**；改規格時記得同步改 `OUTPUT_PATH` 檔名，避免蓋掉別組結果。


In [36]:
# === Standardised barcode table (05_barcodes.csv) ===
# Hard gate: what is written must match what the parameters claim. A length or
# alphabet mismatch raises instead of producing a silently mislabelled file.
observed_lengths = sorted(barcodes['barcode'].str.len().unique())
if observed_lengths != [BARCODE_LEN]:
    raise ValueError(
        f'barcode length mismatch: parameters say {BARCODE_LEN} bp '
        f'({SPEC_NAME}), generated {observed_lengths}'
    )
if len(barcodes) != N_BARCODES:
    raise ValueError(f'expected {N_BARCODES} barcodes, got {len(barcodes)}')
if not barcodes['barcode'].str.fullmatch('[ACGT]+').all():
    raise ValueError('non-ACGT characters in generated barcodes')
if barcodes['barcode'].duplicated().any():
    raise ValueError('duplicate barcodes generated')

barcodes = barcodes.copy()
barcodes.insert(0, 'candidate_id', [f'BC-{i:05d}' for i in range(1, len(barcodes) + 1)])
barcodes['barcode_length'] = BARCODE_LEN
barcodes['min_hamming'] = MIN_HAMMING
barcodes['spec_name'] = SPEC_NAME

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
barcodes.to_csv(OUTPUT_PATH, index=False)
print(f'saved -> {OUTPUT_PATH}')
print(f'spec: {SPEC_NAME}  |  {len(barcodes)} barcodes x {BARCODE_LEN} bp, HD >= {MIN_HAMMING}')
print(barcodes.head(3).to_string(index=False))


saved -> ..\tables\barcodes_15bp_HD4_no_RE_sites.csv
